# Part 2 - SQL: verify the numbers and find where no-shows cluster
**Project: CareLoop Clinics - No-Show Guard**

Numbers that a manager acts on must be reproducible from the database, and they must agree
with the Python numbers from Part 1. In this notebook we load Part 1's output into a database
called `noshowguard` and answer business questions with SQL.

| Step | What we do | SQL ideas |
|---|---|---|
| 2.1 | Create the tables, load the data, test the primary key | `CREATE TABLE`, `PRIMARY KEY`, `FOREIGN KEY` |
| 2.2 | No-show rate per clinic | `JOIN`, `GROUP BY`, `HAVING` |
| 2.3 | No-show rate by lead-time band | `CASE WHEN` |
| 2.4 | Month-by-month trend | CTE (`WITH`), `LAG()` |
| 2.5 | Rank clinics inside each city | CTE, `RANK() OVER (PARTITION BY ...)`, scalar subquery |

Every query here is also saved in `sql/02_queries.sql`.

## Setup
The notebook talks to a **local MySQL 8 server**. The connection details are read from
environment variables (or from a `.env` file that is *not* uploaded to GitHub), so no password
is ever written in this notebook.

If no MySQL server can be reached (for example on Google Colab), the notebook falls back to
**SQLite**, a small database that lives inside Python. Every query below runs on both.

In [1]:
# Install the libraries this notebook needs (safe to re-run).
%pip install -q --disable-pip-version-check pandas mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sqlite3
from decimal import Decimal
from pathlib import Path

import pandas as pd

# On Google Colab the project files are not there yet, so we download (clone) the repo first.
if "google.colab" in str(get_ipython()) and not Path("MASAI-CAPSTONE-PROJ").exists():
    os.system("git clone -q https://github.com/Advay4706/MASAI-CAPSTONE-PROJ")
if Path("MASAI-CAPSTONE-PROJ").exists():
    os.chdir("MASAI-CAPSTONE-PROJ")

BASE = Path(".") if Path("data").exists() else Path("..")   # the project's top folder
OUT = BASE / "outputs"


def load_env_file(path):
    """Read KEY=VALUE lines from a .env file into environment variables (if the file exists)."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        if "=" in line and not line.strip().startswith("#"):
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())


load_env_file(BASE / ".env")

# Try MySQL first. If anything goes wrong (no password set, no server), use SQLite instead.
try:
    import mysql.connector
    conn = mysql.connector.connect(
        host=os.environ.get("MYSQL_HOST", "127.0.0.1"),
        port=int(os.environ.get("MYSQL_PORT", "3306")),
        user=os.environ.get("MYSQL_USER", "root"),
        password=os.environ["MYSQL_PASSWORD"],     # never typed into the notebook
        connection_timeout=5,
    )
    USING = "MySQL"
    DB_ERRORS = (mysql.connector.Error,)
except Exception as err:                           # print only the error type, never its details
    print(f"MySQL not available ({type(err).__name__}) -> using SQLite instead")
    conn = sqlite3.connect(":memory:")             # a temporary database in memory
    USING = "SQLite"
    DB_ERRORS = (sqlite3.Error,)


def run(sql):
    """Run one statement that returns no rows (CREATE, INSERT, ...)."""
    cursor = conn.cursor()
    cursor.execute(sql)
    conn.commit()
    cursor.close()


def tidy(value):
    """MySQL returns exact Decimal numbers; turn them into plain int/float for display."""
    if isinstance(value, Decimal):
        return int(value) if value.as_tuple().exponent >= 0 else float(value)
    return value


def query(sql):
    """Run a SELECT and return the result as a pandas DataFrame."""
    cursor = conn.cursor()
    cursor.execute(sql)
    columns = [column[0] for column in cursor.description]
    rows = [[tidy(v) for v in row] for row in cursor.fetchall()]
    cursor.close()
    return pd.DataFrame(rows, columns=columns)


print("Database engine in use:", USING)
print("Server version:", query("SELECT VERSION() AS v" if USING == "MySQL" else "SELECT sqlite_version() AS v")["v"][0])

Database engine in use: MySQL
Server version: 8.4.9


## Step 2.1 - Build and load the database

**What:** create the database `noshowguard`, create two tables, run `load_data.sql` from
Part 1 (998 `INSERT` statements), and check the counts. Then try to insert a second row with
an `appointment_id` that already exists and record the error.

**Why:** the table definitions are the database's own quality rules.

- `PRIMARY KEY` on `appointment_id` = every appointment has one unique, non-empty ID. The
  database itself refuses a duplicate - we do not have to remember to check.
- `FOREIGN KEY (clinic_id) REFERENCES clinics(clinic_id)` = an appointment can only point to a
  clinic that exists. That is why clinics are loaded first.
- `NOT NULL` = this column may not be empty.
- Types: `VARCHAR(n)` text up to n characters, `INT` whole number, `DATE` a calendar date,
  `DECIMAL(5,1)` an exact number with 5 digits in total and 1 after the decimal point.

In [3]:
CREATE_CLINICS = """
CREATE TABLE clinics (
  clinic_id VARCHAR(5) PRIMARY KEY, clinic_name VARCHAR(40) NOT NULL, city VARCHAR(20) NOT NULL,
  tier VARCHAR(10), latitude DECIMAL(8,4), longitude DECIMAL(8,4), temp_c DECIMAL(5,1));
"""

CREATE_APPOINTMENTS = """
CREATE TABLE appointments (
  appointment_id VARCHAR(8) PRIMARY KEY, patient_id VARCHAR(8) NOT NULL, clinic_id VARCHAR(5) NOT NULL,
  booked_on DATE, appt_date DATE, age INT, gender VARCHAR(1), distance_km DECIMAL(5,1), fee_inr INT,
  booking_channel VARCHAR(10), visit_type VARCHAR(12), reminder_sent VARCHAR(3), prior_no_shows INT,
  status VARCHAR(10), lead_days INT, no_show INT,
  FOREIGN KEY (clinic_id) REFERENCES clinics(clinic_id));
"""

if USING == "MySQL":
    # Start from an empty database so this notebook can be re-run from the top.
    run("DROP DATABASE IF EXISTS noshowguard")
    run("CREATE DATABASE noshowguard")
    run("USE noshowguard")

run(CREATE_CLINICS)          # parent table first
run(CREATE_APPOINTMENTS)     # child table second (it refers to clinics)
print("Tables created.")

Tables created.


In [4]:
# Run load_data.sql: one INSERT statement per line, clinics first.
statements = [line for line in (OUT / "load_data.sql").read_text(encoding="utf-8").splitlines()
              if line.startswith("INSERT INTO")]

cursor = conn.cursor()
for statement in statements:
    cursor.execute(statement)
conn.commit()                # make the inserts permanent
cursor.close()
print("INSERT statements run:", len(statements))

INSERT statements run: 998


In [5]:
Q_COUNTS = """
SELECT
  (SELECT COUNT(*) FROM clinics)          AS clinics,
  (SELECT COUNT(*) FROM appointments)     AS appointments,
  (SELECT SUM(no_show) FROM appointments) AS no_shows;
"""

counts = query(Q_COUNTS)
display(counts)

,clinics,appointments,no_shows
0,8,990,218


### Test the primary key
`A0001` is already in the table. Inserting another row with the same ID must fail.
We catch the error with `try/except` so the notebook keeps running, and print the message.

In [6]:
Q_DUPLICATE = """
INSERT INTO appointments VALUES
  ('A0001','P999','C01','2025-01-01','2025-01-02',30,'F',5.0,500,'App','Consultation','Yes',0,'Attended',1,0);
"""

try:
    run(Q_DUPLICATE)
    print("The insert worked - that would be a problem!")
except DB_ERRORS as err:
    conn.rollback()                      # undo the failed statement
    print("Insert refused.")
    print("Error type:", type(err).__name__)
    print("Error text:", err)

print("\nRows in appointments after the failed insert:",
      query("SELECT COUNT(*) AS n FROM appointments")["n"][0])

Insert refused.
Error type: IntegrityError
Error text: 1062 (23000): Duplicate entry 'A0001' for key 'appointments.PRIMARY'

Rows in appointments after the failed insert: 990


**Reading the result**

- 8 clinics, 990 appointments, 218 no-shows - exactly the numbers from Part 1 (Step 1.4).
  Two different tools agree, which is what makes the numbers trustworthy.
- The duplicate insert fails with MySQL error **1062, "Duplicate entry 'A0001' for key
  'appointments.PRIMARY'"** and the table still has 990 rows. (SQLite words the same refusal
  as "UNIQUE constraint failed".) The same rule is what protects the table against the
  duplicates we had to remove by hand in Step 1.2.

## Step 2.2 - No-show rate by clinic

**What:** join the two tables and, for each clinic, count appointments and no-shows and
compute the no-show percentage. First keep only clinics with at least 95 appointments, then
look at all 8.

**How to read the query**

- `JOIN ... ON a.clinic_id = c.clinic_id` glues each appointment to its clinic row, so we can
  show the clinic's name and city instead of a code.
- `GROUP BY` makes one output row per clinic. `COUNT(*)` counts its rows; `SUM(no_show)` adds
  up the 1s, which is the number of no-shows.
- `100.0 * ...` (with the `.0`) forces decimal division. In some databases `39 / 143` with two
  whole numbers would give `0`.
- `WHERE` filters **rows before** grouping; `HAVING` filters **groups after** grouping. "At least
  95 appointments" is a fact about a group, so it must be `HAVING`.

In [7]:
Q_CLINIC_HAVING = """
SELECT c.clinic_name,
       c.city,
       COUNT(*)                                   AS appointments,
       SUM(a.no_show)                             AS no_shows,
       ROUND(100.0 * SUM(a.no_show) / COUNT(*), 1) AS no_show_pct
FROM appointments a
JOIN clinics c ON a.clinic_id = c.clinic_id
GROUP BY c.clinic_id, c.clinic_name, c.city
HAVING COUNT(*) >= 95
ORDER BY no_show_pct DESC;
"""

clinic_95 = query(Q_CLINIC_HAVING)
display(clinic_95)

,clinic_name,city,appointments,no_shows,no_show_pct
0,Hadapsar Care Centre,Pune,143,39,27.3
1,Malviya Nagar Clinic,Jaipur,142,38,26.8
2,Kothrud Family Clinic,Pune,160,36,22.5
3,Vijay Nagar Clinic,Indore,101,21,20.8
4,Edappally Medical Centre,Kochi,142,23,16.2
5,Vaishali Nagar Clinic,Jaipur,130,18,13.8


### The same query without `HAVING`

In [8]:
Q_CLINIC_ALL = """
SELECT c.clinic_name,
       c.city,
       COUNT(*)                                   AS appointments,
       SUM(a.no_show)                             AS no_shows,
       ROUND(100.0 * SUM(a.no_show) / COUNT(*), 1) AS no_show_pct
FROM appointments a
JOIN clinics c ON a.clinic_id = c.clinic_id
GROUP BY c.clinic_id, c.clinic_name, c.city
ORDER BY no_show_pct DESC;
"""

clinic_all = query(Q_CLINIC_ALL)
display(clinic_all)

top = clinic_all.iloc[0]
print(f"Highest rate: {top['clinic_name']} ({top['city']}) = {top['no_show_pct']}% "
      f"on only {top['appointments']} appointments")

,clinic_name,city,appointments,no_shows,no_show_pct
0,Gomti Nagar Care Point,Lucknow,90,30,33.3
1,Hadapsar Care Centre,Pune,143,39,27.3
2,Malviya Nagar Clinic,Jaipur,142,38,26.8
3,Kothrud Family Clinic,Pune,160,36,22.5
4,Vijay Nagar Clinic,Indore,101,21,20.8
5,Edappally Medical Centre,Kochi,142,23,16.2
6,Sitabuldi Health Point,Nagpur,82,13,15.9
7,Vaishali Nagar Clinic,Jaipur,130,18,13.8


Highest rate: Gomti Nagar Care Point (Lucknow) = 33.3% on only 90 appointments


### Confirm with pandas
The same counts, computed a completely different way: read the CSV files and use `groupby`.
If SQL and pandas agree exactly, a mistake in either one is very unlikely.

In [9]:
appts = pd.read_csv(OUT / "appointments_clean.csv")
clinics = pd.read_csv(OUT / "clinics_enriched.csv")

pandas_counts = (
    appts.merge(clinics, on="clinic_id")                 # same idea as the SQL JOIN
         .groupby("clinic_name")["no_show"]
         .agg(appointments="count", no_shows="sum")      # same idea as COUNT(*) and SUM(no_show)
)
pandas_counts["no_show_pct"] = (100 * pandas_counts["no_shows"] / pandas_counts["appointments"]).round(1)

# Put the SQL result in the same shape and order, then compare.
sql_counts = clinic_all.set_index("clinic_name")[["appointments", "no_shows", "no_show_pct"]]
pandas_counts = pandas_counts.loc[sql_counts.index]

display(pandas_counts)
print("Appointment counts match SQL:", (pandas_counts["appointments"] == sql_counts["appointments"]).all())
print("No-show counts match SQL    :", (pandas_counts["no_shows"] == sql_counts["no_shows"]).all())

,appointments,no_shows,no_show_pct
clinic_name,,,
Gomti Nagar Care Point,90,30,33.3
Hadapsar Care Centre,143,39,27.3
Malviya Nagar Clinic,142,38,26.8
Kothrud Family Clinic,160,36,22.5
Vijay Nagar Clinic,101,21,20.8
Edappally Medical Centre,142,23,16.2
Sitabuldi Health Point,82,13,15.9
Vaishali Nagar Clinic,130,18,13.8


Appointment counts match SQL: True
No-show counts match SQL    : True


**Reading the result**

- Among clinics with at least 95 appointments, **Hadapsar Care Centre (Pune) is worst at 27.3%**
  (39 of 143), closely followed by Malviya Nagar Clinic (Jaipur) at 26.8%. Vaishali Nagar
  Clinic (Jaipur) is best at 13.8%.
- Without `HAVING`, **Gomti Nagar Care Point (Lucknow) tops the list at 33.3% - but on only
  90 appointments.** Sitabuldi Health Point (Nagpur) also appears, with 82 appointments at 15.9%.
- **Why the 95 cut-off matters:** a rate from a small group moves a lot by chance. Three more or
  fewer no-shows change Gomti Nagar's rate by over 3 points. So it is a clinic to *watch*,
  not yet proof that it is the worst.
- The pandas counts match the SQL counts exactly.